<a href="https://colab.research.google.com/github/Komil-jon/ev-battery-vision-pipeline/blob/main/notebooks/colab_train_detector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Train the EV battery detector on Colab (GPU)

Trains YOLOv8n (and optionally YOLO11n) on the **deduped 4,425-image** module/busbar
dataset assembled from ~9 independent Roboflow sources (0 cross-source duplicates,
0 test-set leakage). Runs in ~15-25 min on a Colab GPU vs ~4-5h on CPU.

**Before running:** set Runtime -> Change runtime type -> **GPU (T4/L4/A100)**.
With Colab Pro, pick an L4 or A100 for speed.

**Data:** upload `ev_detector_data.zip` (in your ~/Downloads) to your Google Drive
first (e.g. to `MyDrive/`), then run the cells below.

In [5]:
# 1. GPU check + install
import torch, subprocess
print('CUDA available:', torch.cuda.is_available())
print(subprocess.getoutput('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader'))
!pip install -q ultralytics

CUDA available: True
Tesla T4, 15360 MiB


In [7]:
# 2. Get the data from Google Drive and unzip
from google.colab import drive
drive.mount('/content/drive')

ZIP = '/content/drive/MyDrive/Colab Notebooks/ev_detector_data.zip'
!unzip -q -o "$ZIP" -d /content/
!ls /content/ev_detector_data
!echo "train images:" $(ls /content/ev_detector_data/images/train/*.jpg | wc -l)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
dataset.yaml  images  labels
train images: 4425


In [ ]:
# 3. Train YOLOv8n (paper-style safe augmentation, GPU)
from ultralytics import YOLO

YAML = '/content/ev_detector_data/dataset.yaml'
model = YOLO('yolov8n.pt')
model.train(
    data=YAML, epochs=100, imgsz=640, batch=32, device=0,
    optimizer='SGD', lr0=0.01, weight_decay=0.0005,
    hsv_v=0.30, hsv_s=0.25, hsv_h=0.0,     # no hue jitter (paper: preserves condition cues)
    mosaic=0.5, degrees=2.0, translate=0.06, fliplr=0.5,
    patience=20, project='/content/runs', name='yolov8n_4425', exist_ok=True,
)

Ultralytics 8.4.103 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/ev_detector_data/dataset.yaml, degrees=2.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.0, hsv_s=0.25, hsv_v=0.3, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=0.5, multi_scale=0.0, name=yolov8n_4425, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD,

In [ ]:
# 4. Evaluate on the held-out real test split
metrics = model.val(data=YAML, split='test', imgsz=640, device=0,
                    project='/content/runs', name='yolov8n_4425_test', exist_ok=True)
print(f"TEST mAP50    = {metrics.box.map50:.3f}")
print(f"TEST mAP50-95 = {metrics.box.map:.3f}")
print("(baseline on the paper's 1,759-image data was mAP50 0.818 / mAP50-95 0.557)")

In [ ]:
# 5. (Optional) Same-budget YOLO11n comparison — one line changes
m11 = YOLO('yolo11n.pt')
m11.train(data=YAML, epochs=100, imgsz=640, batch=32, device=0,
          optimizer='SGD', lr0=0.01, weight_decay=0.0005,
          hsv_v=0.30, hsv_s=0.25, hsv_h=0.0, mosaic=0.5, degrees=2.0,
          translate=0.06, fliplr=0.5, patience=20,
          project='/content/runs', name='yolo11n_4425', exist_ok=True)
m11_metrics = m11.val(data=YAML, split='test', imgsz=640, device=0,
                      project='/content/runs', name='yolo11n_4425_test', exist_ok=True)
print(f"YOLO11n TEST mAP50 = {m11_metrics.box.map50:.3f}  mAP50-95 = {m11_metrics.box.map:.3f}")

In [ ]:
# 6. Download the trained weights (and copy to Drive for safekeeping)
from google.colab import files
best = '/content/runs/yolov8n_4425/weights/best.pt'
!cp "$best" /content/drive/MyDrive/ev_detector_yolov8n_4425_best.pt
print('Saved to Drive: MyDrive/ev_detector_yolov8n_4425_best.pt')
files.download(best)

## Back on your machine
Put the downloaded `best.pt` at
`models/detector/stage2_recall_boost/weights/best.pt` to make it the shipped
detector, then run `python scripts/evaluate.py` locally to confirm, and
`python scripts/eval_cross_variant.py` to re-check unseen-pack generalization.